# Music Annotation Project V7.4.1: Optimised Tonal Separation and Structure-Preservation Benchmark

**Runtime:** CPU. Select **Runtime > Run all**.

V7.4.1 is computationally optimised but experimentally identical to V7.4: same methods, thresholds, metrics, masks, Telea baseline and decision logic.

Optimisations only:
- Phase 0 threshold sweeps use 256-bin histograms and cumulative sums rather than rescanning full pages for every threshold;
- observed background normalisation is computed once per page and shared by all tonal candidates;
- clean ink/edge/staff maps are computed once per page during evaluation;
- difference, SSIM, reconstruction ink/edge and tolerance-match maps are computed once per method and reused for regions and connected components;
- staff geometry consumes precomputed ink maps rather than thresholding images again.

Clean pages remain unavailable to candidate reconstruction functions. Pixels outside the removal mask remain immutable.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas matplotlib


## Configuration and cache validation


In [ ]:
import cv2, json, hashlib, shutil, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from skimage.metrics import structural_similarity as ssim

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'; OBSERVED_DIR=ROOT/'aligned'
V71_ROOT=ROOT/'reconstruction_benchmark_v7_1_real'; STATE_PATH=V71_ROOT/'benchmark_state.json'
OUTPUT_ROOT=ROOT/'reconstruction_tonal_benchmark_v7_4'; CACHE=OUTPUT_ROOT/'persistent_cache'
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v7_4'; OUT=OUTPUT_ROOT/'runs'/RUN
TABLES=OUT/'tables'; CHARTS=OUT/'charts'; CONTACTS=OUT/'contact_sheets'; FAILURES=OUT/'failure_crops'
LOCAL=Path('/content/reconstruction_tonal_v7_4'); LCLEAN=LOCAL/'clean'; LOBS=LOCAL/'observed'
for d in [OUTPUT_ROOT,CACHE,OUT,TABLES,CHARTS,CONTACTS,FAILURES,LCLEAN,LOBS]: d.mkdir(parents=True,exist_ok=True)

DILATION=2; RAW_THRESHOLDS=[80,110,140]; NORMALISED_THRESHOLDS=[80,110,140]
FIXED_RAW=110; FIXED_NORM=110; SOFT_LOW=80; SOFT_HIGH=170
BLACK_CORE_RAW=90; BLACK_CORE_NORM=95; AMBIGUOUS_RAW=(75,155); AMBIGUOUS_NORM=(80,165)
BACKGROUND_SIGMA=75.0; CROP_MARGIN=48; BOOTSTRAP_SAMPLES=2000
MIN_COMPONENT_AREA=20; MAX_CONTACT_PAGES=24; MAX_FAILURE_CROPS=20; FORCE_REBUILD=False
METHODS=['telea']+[f'hard_raw_t{t}' for t in RAW_THRESHOLDS]+[f'hard_norm_t{t}' for t in NORMALISED_THRESHOLDS]+['soft_tonal','black_core','triage_hybrid']
SCHEMA='v7.4_tonal_separation_staff_geometry'

def files(folder):
    folder=Path(folder); return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if folder.exists() else []
def maps(folder): return {p.stem:p for p in files(folder)}
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def oh(v): return hashlib.md5(json.dumps(v,sort_keys=True).encode()).hexdigest()
def read_json(p,d=None):
    try:return json.loads(Path(p).read_text()) if Path(p).exists() else d
    except:return d
def valid_npz(p,keys):
    try:
        with np.load(p) as z:return set(keys).issubset(z.files)
    except:return False
def copy_if_needed(a,b):
    b=Path(b)
    if not b.exists() or md5(a)!=md5(b): shutil.copy2(a,b)
def find_local(folder,page):
    h=[p for p in files(folder) if p.stem.lower()==page.lower()]; return h[0] if len(h)==1 else None
def immutable(original,candidate,mask):
    out=original.copy(); out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0]); return out

state=read_json(STATE_PATH)
if not state or not isinstance(state.get('items'),dict): raise RuntimeError(f'Invalid V7.1 state: {STATE_PATH}')
cm,om=maps(CLEAN_DIR),maps(OBSERVED_DIR); pages=[]; issues=[]
for key,item in sorted(state['items'].items()):
    if int(item.get('dilation',-1))!=DILATION: continue
    page=item.get('page'); cache=Path(item.get('reconstruction_cache',''))
    if page not in cm or page not in om or not item.get('reconstruction_complete') or not valid_npz(cache,['mask','telea']):
        issues.append({'item':key,'page':page,'reason':'source or valid d2 cache missing'}); continue
    c=cv2.imread(str(cm[page]),0); o=cv2.imread(str(om[page]),0)
    if c is None or o is None or c.shape!=o.shape:
        issues.append({'item':key,'page':page,'reason':'source unreadable or shape mismatch'}); continue
    with np.load(cache) as z: shape=z['mask'].shape
    if shape!=o.shape: issues.append({'item':key,'page':page,'reason':'cache shape mismatch'}); continue
    copy_if_needed(cm[page],LCLEAN/(page+cm[page].suffix.lower())); copy_if_needed(om[page],LOBS/(page+om[page].suffix.lower())); pages.append(page)
pages=sorted(set(pages)); pd.DataFrame(issues).to_csv(TABLES/'discovery_issues.csv',index=False)
if not pages: raise RuntimeError('No reusable V7.1 dilation-2 caches.')
print('Reusable pages:',len(pages),'Issues:',len(issues))


## Core functions


In [ ]:
def background_normalise(image,sigma=BACKGROUND_SIGMA):
    bg=cv2.GaussianBlur(image,(0,0),sigma); return cv2.divide(image,np.maximum(bg,1),scale=255)
def printed_ink(image): return cv2.adaptiveThreshold(image,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(image): return cv2.Canny(image,70,170)>0
def paper_map(observed,mask):
    result=np.full_like(observed,255)
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    for i in range(1,n):
        x,y,w,h,area=map(int,stats[i]); comp=lab==i
        ring=cv2.dilate(comp.astype(np.uint8),np.ones((21,21),np.uint8))>0; ring&=~comp
        q=observed[ring & (observed>=180)]; value=int(np.median(q)) if q.size else 255; result[comp]=value
    return result

def hard_whiten(observed,mask,paper,signal,threshold):
    out=observed.copy(); whiten=(mask>0)&(signal>=threshold); out[whiten]=paper[whiten]; return immutable(observed,out,mask)
def soft_suppress(observed,mask,paper):
    signal=observed.astype(np.float32); alpha=np.clip((signal-SOFT_LOW)/max(SOFT_HIGH-SOFT_LOW,1),0,1)
    candidate=np.clip(signal*(1-alpha)+paper.astype(np.float32)*alpha,0,255).astype(np.uint8)
    return immutable(observed,candidate,mask)
def black_core_candidate(observed,mask,paper,norm):
    seed=(mask>0)&(observed<=BLACK_CORE_RAW)&(norm<=BLACK_CORE_NORM)
    # Retain only dark evidence with local continuity; isolated dark pencil dots are not privileged.
    support=cv2.dilate(seed.astype(np.uint8),np.ones((3,3),np.uint8))>0
    core=seed & (cv2.boxFilter(seed.astype(np.uint8),-1,(3,3),normalize=False)>=2)
    core=cv2.dilate(core.astype(np.uint8),np.ones((2,2),np.uint8))>0; core&=support&(mask>0)
    out=observed.copy(); out[(mask>0)&~core]=paper[(mask>0)&~core]
    return immutable(observed,out,mask),core

def triage_candidate(observed,mask,telea,paper,norm):
    preserve=(mask>0)&(observed<=AMBIGUOUS_RAW[0])&(norm<=AMBIGUOUS_NORM[0])
    remove=(mask>0)&(observed>=AMBIGUOUS_RAW[1])&(norm>=AMBIGUOUS_NORM[1])
    ambiguous=(mask>0)&~preserve&~remove
    out=observed.copy(); out[remove]=paper[remove]; out[ambiguous]=telea[ambiguous]
    return immutable(observed,out,mask),preserve,remove,ambiguous

def classify(clean,mask):
    ink=printed_ink(clean); long=cv2.morphologyEx(ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long.astype(np.uint8),np.ones((3,7),np.uint8))>0; near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0; m=mask>0
    return {'all':m,'background':m&~ink,'staff':m&staff,'symbol':m&ink&~near,'mixed':m&ink&near},ink,staff


## Phase 0: intensity separability analysis


In [ ]:
separation=[]
histogram={'raw_print':np.zeros(256,np.int64),'raw_paper':np.zeros(256,np.int64),'norm_print':np.zeros(256,np.int64),'norm_paper':np.zeros(256,np.int64)}

def threshold_rows_from_hist(page,signal,print_counts,paper_counts):
    # Threshold semantics remain exactly: dark (< threshold) predicts printed.
    print_cum=np.concatenate(([0],np.cumsum(print_counts,dtype=np.int64)))
    paper_cum=np.concatenate(([0],np.cumsum(paper_counts,dtype=np.int64)))
    print_total=int(print_cum[-1]); paper_total=int(paper_cum[-1]); rows=[]
    for threshold in range(256):
        tp=int(print_cum[threshold]); fn=print_total-tp
        fp=int(paper_cum[threshold]); tn=paper_total-fp
        rows.append({'page':page,'signal':signal,'threshold':threshold,
            'printed_recall':tp/max(tp+fn,1),'printed_precision':tp/max(tp+fp,1),
            'paper_recall':tn/max(tn+fp,1),'balanced_accuracy':.5*(tp/max(tp+fn,1)+tn/max(tn+fp,1)),
            'printed_pixels_whitened':fn,'paper_pixels_preserved_dark':fp})
    return rows

for page in pages:
    clean=cv2.imread(str(find_local(LCLEAN,page)),0); observed=cv2.imread(str(find_local(LOBS,page)),0)
    item=state['items'][f'{page}|d{DILATION}']
    with np.load(item['reconstruction_cache']) as z: mask=z['mask']>0
    clean_ink=printed_ink(clean); truth=clean_ink&mask; paper=mask&~clean_ink
    norm=background_normalise(observed)
    for name,signal in [('raw',observed),('norm',norm)]:
        pc=np.bincount(signal[truth],minlength=256); bc=np.bincount(signal[paper],minlength=256)
        histogram[f'{name}_print']+=pc; histogram[f'{name}_paper']+=bc
        separation.extend(threshold_rows_from_hist(page,name,pc,bc))
sep=pd.DataFrame(separation); sep.to_csv(TABLES/'intensity_threshold_sweep_by_page.csv',index=False)
pooled=[]
for signal in ['raw','norm']:
    pooled.extend(threshold_rows_from_hist('POOLED',signal,histogram[f'{signal}_print'],histogram[f'{signal}_paper']))
pooled=pd.DataFrame(pooled).drop(columns='page'); pooled.to_csv(TABLES/'intensity_threshold_sweep_pooled.csv',index=False)
for signal in ['raw','norm']:
    plt.figure(figsize=(11,6)); x=np.arange(256)
    a=histogram[f'{signal}_print'].astype(float); b=histogram[f'{signal}_paper'].astype(float)
    plt.plot(x,a/max(a.sum(),1),label='clean truth: printed'); plt.plot(x,b/max(b.sum(),1),label='clean truth: paper')
    plt.xlabel('Observed intensity'); plt.ylabel('Proportion'); plt.title(f'{signal} intensity separability inside removal masks'); plt.legend(); plt.tight_layout(); plt.savefig(CHARTS/f'{signal}_intensity_distributions.png',dpi=170); plt.close()
    q=pooled[pooled.signal==signal]; best=q.loc[q.balanced_accuracy.idxmax()]
    print(signal,'exploratory best threshold',int(best.threshold),'balanced accuracy',round(float(best.balanced_accuracy),4))
print('Phase 0 complete. Fixed benchmark thresholds remain unchanged.')


## Construct all candidates without clean-page access


In [ ]:
cache_records=[]; route_records=[]
for index,page in enumerate(pages,1):
    observed=cv2.imread(str(find_local(LOBS,page)),0); item=state['items'][f'{page}|d{DILATION}']
    with np.load(item['reconstruction_cache']) as z: mask=z['mask']; telea=z['telea']
    fp=oh({'v71':item.get('fingerprint'),'observed':md5(find_local(LOBS,page)),'schema':SCHEMA,'raw':RAW_THRESHOLDS,'norm':NORMALISED_THRESHOLDS,
           'soft':[SOFT_LOW,SOFT_HIGH],'core':[BLACK_CORE_RAW,BLACK_CORE_NORM],'triage':[AMBIGUOUS_RAW,AMBIGUOUS_NORM]})
    path=CACHE/f'{page}_{fp}.npz'
    if path.exists() and not FORCE_REBUILD and valid_npz(path,METHODS+['mask','core_mask','triage_preserve','triage_remove','triage_ambiguous']):
        hit=True
    else:
        paper=paper_map(observed,mask); norm=background_normalise(observed); outputs={'telea':telea}
        for t in RAW_THRESHOLDS: outputs[f'hard_raw_t{t}']=hard_whiten(observed,mask,paper,observed,t)
        for t in NORMALISED_THRESHOLDS: outputs[f'hard_norm_t{t}']=hard_whiten(observed,mask,paper,norm,t)
        outputs['soft_tonal']=soft_suppress(observed,mask,paper)
        outputs['black_core'],core=black_core_candidate(observed,mask,paper,norm)
        outputs['triage_hybrid'],preserve,remove,ambiguous=triage_candidate(observed,mask,telea,paper,norm)
        for m,o in outputs.items(): assert np.array_equal(o[mask==0],observed[mask==0])
        np.savez_compressed(path,mask=mask,core_mask=core.astype(np.uint8),triage_preserve=preserve.astype(np.uint8),triage_remove=remove.astype(np.uint8),triage_ambiguous=ambiguous.astype(np.uint8),**outputs); hit=False
    with np.load(path) as z:
        route_records.append({'page':page,'mask_pixels':int(np.sum(z['mask']>0)),'black_core_pixels':int(np.sum(z['core_mask']>0)),
            'triage_preserve_pixels':int(np.sum(z['triage_preserve']>0)),'triage_remove_pixels':int(np.sum(z['triage_remove']>0)),
            'triage_ambiguous_pixels':int(np.sum(z['triage_ambiguous']>0))})
    cache_records.append({'page':page,'cache':str(path),'cache_hit':hit}); print(f'Construct {index}/{len(pages)} {page}:', 'HIT' if hit else 'MISS')
pd.DataFrame(cache_records).to_csv(TABLES/'reconstruction_cache_status.csv',index=False); pd.DataFrame(route_records).to_csv(TABLES/'method_routing_by_page.csv',index=False)


## Evaluation metrics, including staff geometry


In [ ]:
def tolerance_bundle(reference,prediction,tol=1):
    k=np.ones((2*tol+1,2*tol+1),np.uint8)
    rd=cv2.dilate(reference.astype(np.uint8),k)>0; pd=cv2.dilate(prediction.astype(np.uint8),k)>0
    return {'reference':reference,'prediction':prediction,'prediction_match':prediction&rd,'reference_match':reference&pd}

def f1_from_bundle(bundle,region):
    ref_count=int(np.sum(bundle['reference']&region))
    if not ref_count:return np.nan,np.nan,np.nan
    pred_count=int(np.sum(bundle['prediction']&region))
    p=float(np.sum(bundle['prediction_match']&region)/max(pred_count,1)); r=float(np.sum(bundle['reference_match']&region)/ref_count)
    return p,r,float(2*p*r/max(p+r,1e-12))

def method_bundle(clean,recon,clean_ink,clean_edges):
    d=clean.astype(np.float32)-recon.astype(np.float32); _,sm=ssim(clean,recon,data_range=255,full=True)
    reconstruction_ink=printed_ink(recon); reconstruction_edges=edges(recon)
    return {'abs_diff':np.abs(d),'squared_diff':d*d,'ssim_map':sm.astype(np.float32,copy=False),
        'ink':tolerance_bundle(clean_ink,reconstruction_ink),'edge':tolerance_bundle(clean_edges,reconstruction_edges),
        'reconstruction_ink':reconstruction_ink,'dark_smudge':(recon<180)&(clean>=220),'clean_light':clean>=220}

def staff_geometry_from_ink(clean_ink,reconstruction_ink,staff_region):
    ref=clean_ink&staff_region; pred=reconstruction_ink&staff_region; cols=np.where(ref.sum(axis=0)>0)[0]
    if not len(cols):return {'staff_gap_rate':np.nan,'staff_centreline_mae':np.nan,'staff_centreline_max':np.nan,'staff_thickness_mae':np.nan}
    offsets=[]; thickness=[]; gaps=0; valid=0
    for x in cols:
        ry=np.where(ref[:,x])[0]
        if not len(ry):continue
        py=np.where(pred[:,x])[0]; valid+=1
        if not len(py):gaps+=1;continue
        rc=float(np.median(ry)); pc=float(py[np.argmin(np.abs(py-rc))]); offsets.append(abs(pc-rc)); thickness.append(abs(len(py)-len(ry)))
    return {'staff_gap_rate':float(gaps/max(valid,1)),'staff_centreline_mae':float(np.mean(offsets)) if offsets else np.nan,
        'staff_centreline_max':float(np.max(offsets)) if offsets else np.nan,'staff_thickness_mae':float(np.mean(thickness)) if thickness else np.nan}

def score_bundle(bundle,region,clean_ink=None,staff_region=None):
    pixels=int(region.sum())
    if not pixels:return None
    mse=float(np.mean(bundle['squared_diff'][region])); ip,ir,if1=f1_from_bundle(bundle['ink'],region); _,_,ef=f1_from_bundle(bundle['edge'],region)
    out={'pixels':pixels,'mae':float(np.mean(bundle['abs_diff'][region])),'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),
        'ssim':float(np.mean(bundle['ssim_map'][region])),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1_tolerant':ef,
        'dark_smudge_rate':float(np.sum(bundle['dark_smudge']&region)/max(np.sum(bundle['clean_light']&region),1))}
    if staff_region is not None:out.update(staff_geometry_from_ink(clean_ink,bundle['reconstruction_ink'],staff_region))
    return out

rows=[]; component_rows=[]
for index,page in enumerate(pages,1):
    clean=cv2.imread(str(find_local(LCLEAN,page)),0); rec=next(x for x in cache_records if x['page']==page)
    with np.load(rec['cache']) as z: mask=z['mask']; outputs={m:z[m] for m in METHODS}
    regions,clean_ink,staff=classify(clean,mask); clean_edges=edges(clean)
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    component_specs=[]
    for i in range(1,n):
        if stats[i,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
        comp=lab==i; frac=float(np.mean(clean_ink[comp])); category='background' if frac<.05 else ('structure' if frac>.30 else 'mixed')
        component_specs.append((i,comp,category))
    for method,recon in outputs.items():
        bundle=method_bundle(clean,recon,clean_ink,clean_edges)
        for name,region in regions.items():
            q=score_bundle(bundle,region,clean_ink,staff if name=='staff' else None)
            if q:rows.append({'page':page,'method':method,'region':name,**q})
        for i,comp,category in component_specs:
            q=score_bundle(bundle,comp,clean_ink,comp&staff)
            if q:component_rows.append({'page':page,'component':i,'category':category,'component_area':int(stats[i,cv2.CC_STAT_AREA]),
                'x':int(stats[i,0]),'y':int(stats[i,1]),'w':int(stats[i,2]),'h':int(stats[i,3]),'method':method,**q})
        del bundle
    print(f'Score {index}/{len(pages)} {page}')
results=pd.DataFrame(rows); components=pd.DataFrame(component_rows)
results.to_csv(TABLES/'metrics_by_page_region.csv',index=False); components.to_csv(TABLES/'component_metrics.csv',index=False)


## Paired comparisons against Telea and structural decision table


In [ ]:
def boot(v,seed=42):
    v=np.asarray(v,float); v=v[np.isfinite(v)]
    if not len(v): return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed); s=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(BOOTSTRAP_SAMPLES)])
    return float(v.mean()),float(np.percentile(s,2.5)),float(np.percentile(s,97.5))
specs=[('background','mae',False),('background','dark_smudge_rate',False),('staff','ink_recall',True),('staff','staff_gap_rate',False),
       ('staff','staff_centreline_mae',False),('staff','staff_centreline_max',False),('staff','staff_thickness_mae',False),
       ('symbol','edge_f1_tolerant',True),('mixed','edge_f1_tolerant',True),('all','mae',False),('all','ssim',True)]
paired=[]
for region,metric,higher in specs:
    p=results[results.region==region].pivot(index='page',columns='method',values=metric)
    if 'telea' not in p: continue
    for method in METHODS:
        if method=='telea' or method not in p: continue
        d=(p[method]-p.telea)*(1 if higher else -1); mean,lo,hi=boot(d.dropna())
        paired.append({'region':region,'metric':metric,'method':method,'baseline':'telea','benefit_mean':mean,'ci95_low':lo,'ci95_high':hi,
            'wins':int((d>0).sum()),'losses':int((d<0).sum()),'ties':int((d==0).sum()),'pages':int(d.notna().sum())})
paired=pd.DataFrame(paired); paired.to_csv(TABLES/'paired_vs_telea.csv',index=False)
summary=results.groupby(['method','region'],as_index=False).agg(pages=('page','nunique'),pixels=('pixels','sum'),mae=('mae','mean'),ssim=('ssim','mean'),
    ink_recall=('ink_recall','mean'),edge_f1_tolerant=('edge_f1_tolerant','mean'),dark_smudge_rate=('dark_smudge_rate','mean'),staff_gap_rate=('staff_gap_rate','mean'),
    staff_centreline_mae=('staff_centreline_mae','mean'),staff_centreline_max=('staff_centreline_max','mean'),staff_thickness_mae=('staff_thickness_mae','mean'))
summary.to_csv(TABLES/'summary_by_region.csv',index=False)
# Conservative ranking: any statistically supported structural degradation blocks candidacy.
def pv(method,region,metric,col):
    q=paired[(paired.method==method)&(paired.region==region)&(paired.metric==metric)]; return float(q.iloc[0][col]) if len(q) else np.nan
dec=[]
for m in METHODS[1:]:
    staff_recall_low=pv(m,'staff','ink_recall','ci95_low'); gap_low=pv(m,'staff','staff_gap_rate','ci95_low'); centre_low=pv(m,'staff','staff_centreline_mae','ci95_low'); symbol_low=pv(m,'symbol','edge_f1_tolerant','ci95_low')
    bg_low=pv(m,'background','mae','ci95_low')
    safe=bool(staff_recall_low>=-0.005 and gap_low>=-0.005 and centre_low>=-0.10 and symbol_low>=-0.010)
    dec.append({'method':m,'background_mae_ci95_low':bg_low,'staff_recall_ci95_low':staff_recall_low,'staff_gap_benefit_ci95_low':gap_low,
        'staff_centreline_benefit_ci95_low':centre_low,'symbol_edge_ci95_low':symbol_low,'structurally_noninferior':safe,
        'background_improvement_supported':bool(bg_low>0),'provisional_candidate':bool(safe and bg_low>0)})
decision=pd.DataFrame(dec); decision.to_csv(TABLES/'structural_decision_table.csv',index=False)
display(decision.sort_values(['provisional_candidate','structurally_noninferior','background_mae_ci95_low'],ascending=False))


## Contact sheets and worst structural crops


In [ ]:
priority=components[components.method=='telea'].sort_values(['staff_centreline_max','component_area'],ascending=False).page.drop_duplicates().head(MAX_CONTACT_PAGES)
show_methods=['telea','hard_raw_t110','hard_norm_t110','soft_tonal','black_core','triage_hybrid']
for page in priority:
    clean=cv2.imread(str(find_local(LCLEAN,page)),0); observed=cv2.imread(str(find_local(LOBS,page)),0); rec=next(x for x in cache_records if x['page']==page)
    with np.load(rec['cache']) as z: imgs=[clean,observed]+[z[m] for m in show_methods]
    names=['clean','observed']+show_methods; fig,axes=plt.subplots(2,4,figsize=(24,14))
    for ax,im,name in zip(axes.ravel(),imgs,names): ax.imshow(im,cmap='gray',vmin=0,vmax=255); ax.set_title(name); ax.axis('off')
    plt.tight_layout(); plt.savefig(CONTACTS/f'{page}_comparison.png',dpi=150,bbox_inches='tight'); plt.close(fig)
for metric in ['staff_gap_rate','staff_centreline_max','staff_thickness_mae']:
    folder=FAILURES/metric; folder.mkdir(parents=True,exist_ok=True)
    target=components[(components.category!='background')&(components.method=='triage_hybrid')].dropna(subset=[metric]).sort_values(metric,ascending=False).head(MAX_FAILURE_CROPS)
    for rank,(_,r) in enumerate(target.iterrows(),1):
        page=str(r.page); clean=cv2.imread(str(find_local(LCLEAN,page)),0); observed=cv2.imread(str(find_local(LOBS,page)),0); rec=next(x for x in cache_records if x['page']==page)
        with np.load(rec['cache']) as z: imgs=[clean,observed]+[z[m] for m in show_methods]
        p=30; H,W=clean.shape; x0=max(0,int(r.x)-p);y0=max(0,int(r.y)-p);x1=min(W,int(r.x+r.w)+p);y1=min(H,int(r.y+r.h)+p)
        fig,axes=plt.subplots(2,4,figsize=(20,9)); names=['clean','observed']+show_methods
        for ax,im,name in zip(axes.ravel(),imgs,names): ax.imshow(im[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255);ax.set_title(name);ax.axis('off')
        plt.tight_layout();plt.savefig(folder/f'{rank:03d}_{page}_c{int(r.component)}.png',dpi=170,bbox_inches='tight');plt.close(fig)
print('Contact sheets:',CONTACTS,'Failures:',FAILURES)


## Manifest


In [ ]:
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.4.1','pages':len(pages),'dilation':DILATION,'methods':METHODS,'schema':SCHEMA,
 'settings':{'raw_thresholds':RAW_THRESHOLDS,'normalised_thresholds':NORMALISED_THRESHOLDS,'fixed_raw':FIXED_RAW,'fixed_norm':FIXED_NORM,
 'soft':[SOFT_LOW,SOFT_HIGH],'black_core':[BLACK_CORE_RAW,BLACK_CORE_NORM],'triage_raw':AMBIGUOUS_RAW,'triage_norm':AMBIGUOUS_NORM,
 'background_sigma':BACKGROUND_SIGMA,'bootstrap_samples':BOOTSTRAP_SAMPLES},
 'guarantees':{'clean_used_for_phase0_analysis_and_scoring_only':True,'candidate_functions_do_not_receive_clean':True,'outside_mask_immutable_asserted':True,
 'v7_1_dilation2_mask_and_telea_reused':True,'exploratory_best_threshold_not_auto_selected':True,'phase0_histogram_cumulative_sweep':True,'evaluation_feature_maps_precomputed_once_per_method':True},
 'warning':'Exploratory threshold comparison on the existing controlled population. Freeze candidate settings before independent confirmation.'}
(OUT/'benchmark_manifest.json').write_text(json.dumps(manifest,indent=2))
print('V7.4.1 COMPLETE:',OUT)
